In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# 3-Class Random Forest ESI 2/3 vs ESI 4/5 vs Other Classifier (`models/rf_esi23_esi45_extreme.ipynb`)

This notebook trains a **3-Class Random Forest Model** for **ESI 2 & 3 vs ESI 4 & 5 vs Other** (where ESI 1 cases map to `"other"`) using **17 Predictor Features** (7 raw triage inputs + 10 binary vital anomaly flags; strictly excluding non-binary feature engineered deltas/ranges and vital history min/max/last columns), **Factor-Controlled Minority Class Random Upsampling**, 5-Fold Stratified Cross-Validation, **Balanced Accuracy**, **Specificity**, and **MCC**:

### System Architecture & Workflow
1. **3-Class Target Definition (`"2_3"`, `"4_5"`, `"other"`)**:
   - ESI 2 or ESI 3 patients $\rightarrow$ Class `"2_3"`
   - ESI 4 or ESI 5 patients $\rightarrow$ Class `"4_5"`
   - ESI 1 patients $\rightarrow$ Class `"other"`
2. **Predictor Feature Selection (17 Total Features)**:
   - **7 Raw Triage Inputs**: `age`, `gender`, `cc_breathingdifficulty`, `triage_vital_hr`, `triage_vital_sbp`, `triage_vital_rr`, `triage_vital_o2`.
   - **10 Binary Vital Anomaly Flags**: `is_dyspnea_total`, `is_dyspnea_moderate`, `is_bradypnea`, `is_tachypnea`, `is_hypotension`, `is_hypertension`, `is_bradycardia_total`, `is_bradycardia_moderate`, `is_tachycardia_total`, `is_tachycardia_moderate`.
3. **Stratified Data Partitioning & Oversampling**:
   - Reserves a **15% Holdout Test Set** evaluated ONCE at the end.
   - Evaluates the **85% Validation Set** via **5-Fold Stratified Cross-Validation**.
   - Applies **Minority Class Bootstrap Random Upsampling** (`upsample_ratio = 1.0`) strictly to training partitions.
4. **Random Forest Ensemble (`ranger`)**: Fits decision trees with `probability = TRUE`.
5. **Full Metrics Suite Evaluation**: Computes Per-Class and Macro-Averaged Accuracy, **Balanced Accuracy**, **Specificity**, Precision, Recall/Sensitivity, F1 Score, ROC-AUC, and **MCC Score**.
6. **Reports & Artifacts**:
   - **CSV Reports**: `reports/rf_esi23_esi45_5fold_cv_report.csv` and `reports/rf_esi23_esi45_test_report.csv`.
   - **Diagnostic Plots**: Metrics bar chart (`plots/rf_esi23_esi45_metrics_barchart.png`).
   - **Model Export**: Saved to `deploy/rf_esi23_esi45_extreme_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(ranger)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Construct 17 Features & Define 3-Class Target Target
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))
raw_df <- get(data_obj_name, envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
t_hr  <- get_vec("triage_vital_hr")
t_sbp <- get_vec("triage_vital_sbp")
t_o2  <- get_vec("triage_vital_o2")
t_rr  <- get_vec("triage_vital_rr")
# Construct 17 Predictor Features (7 raw triage + 10 binary vital anomaly flags)
df_full <- data.frame(
  # 7 Raw Triage Inputs
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  triage_vital_hr         = t_hr,
  triage_vital_sbp        = t_sbp,
  triage_vital_rr         = t_rr,
  triage_vital_o2         = t_o2,
  
  # 10 Binary Vital Anomaly Flags
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)
# Target Mapping: ESI 2/3 -> '2_3', ESI 4/5 -> '4_5', ESI 1 -> 'other'
raw_esi <- as.character(raw_df[[target_col_name]])
target_vec <- ifelse(raw_esi %in% c("2", "3"), "2_3", ifelse(raw_esi %in% c("4", "5"), "4_5", "other"))
df_full$target_col <- factor(target_vec, levels = c("2_3", "4_5", "other"))
initial_rows <- nrow(df_full)
df_full <- na.omit(df_full)
cat(sprintf("Complete Case Filtering: Removed %d rows (Remaining: %d)\n", initial_rows - nrow(df_full), nrow(df_full)))
cat(sprintf("Full 3-Class Dataset Ready (17 Features): %d total rows x %d cols\n", nrow(df_full), ncol(df_full)))
print(table(df_full$target_col))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Partitioning & 5-Fold CV Random Forest
# ---------------------------------------------------------
set.seed(config$training$random_state)
test_size <- config$training$test_size
in_train_val <- createDataPartition(df_full$target_col, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
upsample_ratio <- 1.0
upsample_multiclass <- function(df_train, ratio = 1.0) {
  counts <- table(df_train$target_col)
  max_cnt <- max(counts)
  
  res_df <- df_train
  for (cls in names(counts)) {
    cls_cnt <- counts[[cls]]
    target_cnt <- round(max_cnt * ratio)
    if (target_cnt > cls_cnt) {
      extra_needed <- target_cnt - cls_cnt
      cls_subset   <- df_train[df_train$target_col == cls, ]
      sampled_extra <- cls_subset[sample(1:cls_cnt, size = extra_needed, replace = TRUE), ]
      res_df       <- rbind(res_df, sampled_extra)
    }
  }
  return(res_df)
}
k_folds <- 5
folds   <- createFolds(train_val_df$target_col, k = k_folds, list = TRUE, returnTrain = FALSE)
binary_cols <- c("gender", "cc_breathingdifficulty",
                 "is_dyspnea_total", "is_dyspnea_moderate", "is_bradypnea", "is_tachypnea",
                 "is_hypotension", "is_hypertension", "is_bradycardia_total", "is_bradycardia_moderate",
                 "is_tachycardia_total", "is_tachycardia_moderate")
cont_cols <- setdiff(names(train_val_df), c(binary_cols, "target_col"))
val_fold_accs     <- numeric(k_folds)
val_fold_bal_accs <- numeric(k_folds)
val_fold_specs    <- numeric(k_folds)
for (k in 1:k_folds) {
  val_idx    <- folds[[k]]
  train_fold <- train_val_df[-val_idx, ]
  val_fold   <- train_val_df[val_idx, ]
  
  train_fold <- upsample_multiclass(train_fold, ratio = upsample_ratio)
  
  preproc_fold <- preProcess(train_fold[, cont_cols, drop = FALSE], method = c("center", "scale"))
  train_fold   <- predict(preproc_fold, train_fold)
  val_fold     <- predict(preproc_fold, val_fold)
  
  rf_fold <- ranger(
    formula     = target_col ~ .,
    data        = train_fold,
    num.trees   = 200,
    probability = TRUE,
    seed        = config$training$random_state
  )
  
  probs_mat <- predict(rf_fold, data = val_fold)$predictions
  fold_pred_idx <- apply(probs_mat, 1, which.max)
  fold_pred_fac <- factor(colnames(probs_mat)[fold_pred_idx], levels = c("2_3", "4_5", "other"))
  fold_cm       <- confusionMatrix(fold_pred_fac, val_fold$target_col)
  
  val_fold_accs[k]     <- as.numeric(fold_cm$overall["Accuracy"])
  val_fold_bal_accs[k] <- mean(fold_cm$byClass[, "Balanced Accuracy"])
  val_fold_specs[k]    <- mean(fold_cm$byClass[, "Specificity"])
}
cat(sprintf("5-Fold CV Mean Acc = %.4f | Mean BalAcc = %.4f | Mean Spec = %.4f\n",
            mean(val_fold_accs), mean(val_fold_bal_accs), mean(val_fold_specs)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Final Production Random Forest Training & Test Set Benchmark
# ---------------------------------------------------------
train_val_upsampled <- upsample_multiclass(train_val_df, ratio = upsample_ratio)
preproc_tv <- preProcess(train_val_upsampled[, cont_cols, drop = FALSE], method = c("center", "scale"))
train_val_scaled <- predict(preproc_tv, train_val_upsampled)
test_scaled      <- predict(preproc_tv, test_df)
final_rf <- ranger(
  formula     = target_col ~ .,
  data        = train_val_scaled,
  num.trees   = 250,
  probability = TRUE,
  seed        = config$training$random_state
)
raw_test_probs <- predict(final_rf, data = test_scaled)$predictions
test_pred_idx  <- apply(raw_test_probs, 1, which.max)
test_pred_fac  <- factor(colnames(raw_test_probs)[test_pred_idx], levels = c("2_3", "4_5", "other"))
act_test_fac   <- factor(test_df$target_col, levels = c("2_3", "4_5", "other"))
cm_test  <- confusionMatrix(test_pred_fac, act_test_fac)
acc_test <- as.numeric(cm_test$overall["Accuracy"])
prec_by_class    <- as.numeric(cm_test$byClass[, "Pos Pred Value"])
rec_by_class     <- as.numeric(cm_test$byClass[, "Sensitivity"])
spec_by_class    <- as.numeric(cm_test$byClass[, "Specificity"])
bal_acc_by_class <- as.numeric(cm_test$byClass[, "Balanced Accuracy"])
prec_by_class[is.na(prec_by_class)]       <- 0
rec_by_class[is.na(rec_by_class)]         <- 0
spec_by_class[is.na(spec_by_class)]       <- 0
bal_acc_by_class[is.na(bal_acc_by_class)] <- 0
f1_by_class <- ifelse((prec_by_class + rec_by_class) > 0, 2 * (prec_by_class * rec_by_class) / (prec_by_class + rec_by_class), 0)
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
saveRDS(list(model = final_rf, preproc = preproc_tv, upsample_ratio = upsample_ratio), file = file.path(deploy_dir, "rf_esi23_esi45_extreme_model.rds"))
cat("Final Random Forest Model saved to deploy/rf_esi23_esi45_extreme_model.rds\n")